# Qwen3.5-4B → OpenVINO INT4 Split-IR + HuggingFace Push

Converts Qwen3.5-4B weights to **INT4 quantized split-IR** OpenVINO format.
Each decoder layer is a separate `.xml/.bin` file — low RAM, low disk.

| Step | Action |
|------|--------|
| 0 | Load Kaggle secrets |
| 1 | `git clone dsainvg/openvino-model-conv` |
| 2 | Install requirements (torch CPU + openvino + **nncf** + huggingface_hub) |
| 3 | `pytest qwen35/tests/` — toy smoke tests |
| 4 | `download_model.py` — pull Qwen3.5-4B weights from HF |
| 5 | `convert_to_openvino.py` — INT4 split-IR conversion |
| 6 | `push_to_hf.py` — upload to HuggingFace |

> **Kaggle Secrets needed** (Add-ons → Secrets):
> - `HF_TOKEN` — HuggingFace token with **write** scope
> - `HF_REPO_NAME` — target repo *(default: `qwen35-4b-openvino-int4`)*

---
**Quantization**: NNCF `INT4_ASYM`, `group_size=64`, `ratio=1.0`  
**Expected output size**: ~2–2.5 GB (vs ~9 GB FP16)  
**Expected runtime**: ~90–120 min on Kaggle CPU

```
ov_ir_qwen35_4b_int4/
  embed.xml + embed.bin      ← FP16 (embedding lookup)
  layer_0.xml + layer_0.bin  ← INT4 (linear_attention)
  layer_1.xml + layer_1.bin  ← INT4 (linear_attention)
  layer_2.xml + layer_2.bin  ← INT4 (linear_attention)
  layer_3.xml + layer_3.bin  ← INT4 (full_attention)
  ...  (32 layers total)
  lm_head.xml + lm_head.bin  ← INT4
```

## 0 · Secrets

In [ ]:
import os

try:
    from kaggle_secrets import UserSecretsClient
    _s = UserSecretsClient()
    def _get(key, fallback=None):
        try:
            return _s.get_secret(key)
        except Exception:
            return fallback
except ImportError:
    def _get(key, fallback=None):
        return os.environ.get(key, fallback)

HF_TOKEN     = _get("HF_TOKEN")
HF_REPO_NAME = _get("HF_REPO_NAME", "qwen35-4b-openvino-int4")

if not HF_TOKEN:
    raise EnvironmentError("HF_TOKEN secret is missing. Add it under Add-ons → Secrets.")

os.environ["HF_TOKEN"]               = HF_TOKEN
os.environ["HUGGING_FACE_HUB_TOKEN"] = HF_TOKEN

print(f"HF_REPO_NAME : {HF_REPO_NAME}")
print("HF_TOKEN     : *** (set)")

## 1 · Clone the converter repo

In [ ]:
import subprocess, sys
from pathlib import Path

REPO_URL = "https://github.com/dsainvg/openvino-model-conv.git"
REPO_DIR = Path("/kaggle/working/openvino-model-conv")

if REPO_DIR.exists():
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "--depth=1", REPO_URL, str(REPO_DIR)], check=True)

QWEN35_DIR  = REPO_DIR / "qwen35"
SCRIPTS_DIR = QWEN35_DIR / "scripts"
MODEL_DIR   = Path("/kaggle/working/Qwen3.5-4B")
OUTPUT_DIR  = Path("/kaggle/working/ov_ir_qwen35_4b_int4")

print(f"Repo    : {REPO_DIR}")
print(f"Scripts : {SCRIPTS_DIR}")

## 2 · Install requirements

In [ ]:
def pip(*args):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *args], check=True)

# CPU-only torch to avoid CUDA version conflicts with Kaggle's cuDNN stack
pip("torch", "--index-url", "https://download.pytorch.org/whl/cpu")
pip(
    "openvino",          # ov.convert_model / ov.save_model
    "nncf",              # INT4 weight compression (compress_weights)
    "huggingface_hub",   # hf_hub_download / push_to_hub
    "safetensors",       # safe_open for weight loading
    "sentencepiece",     # tokenizer
    "tiktoken",          # tokenizer (some checkpoints)
    "accelerate",        # optional: used by transformers for fast loading
    "pytest",            # smoke tests
)

print("Done.")

## 3 · Toy smoke tests (`qwen35/tests/`)

Runs all 11 unit + integration tests against tiny random-weight models.  
No downloads needed. Should finish in under 30 seconds.

In [ ]:
result = subprocess.run(
    [sys.executable, "-m", "pytest", "tests/", "-v", "--tb=short"],
    cwd=str(QWEN35_DIR),
    env={**os.environ},
)
if result.returncode != 0:
    raise RuntimeError("Toy smoke tests FAILED — fix modeling code before converting real weights.")
print("\n✓ All toy tests passed.")

## 4 · Download Qwen3.5-4B weights

In [ ]:
result = subprocess.run(
    [
        sys.executable, str(SCRIPTS_DIR / "download_model.py"),
        "--model",  "Qwen/Qwen3.5-4B",
        "--output", str(MODEL_DIR),
        "--token",  HF_TOKEN,
    ],
    cwd=str(QWEN35_DIR),
    env={**os.environ},
)
if result.returncode != 0:
    raise RuntimeError("download_model.py failed.")
print("\n✓ Model downloaded to", MODEL_DIR)

## 5 · Convert to OpenVINO INT4 Split-IR

For each component:
1. Load BF16 weights into the module (one layer at a time)
2. `ov.convert_model(wrapper, example_input=...)` — trace to OV IR
3. `nncf.compress_weights(INT4_ASYM, group_size=64)` — quantize Linear weights to INT4
4. Save `.xml/.bin` and free RAM before next layer

**Embedding** is kept FP16 (INT4 on a lookup table has no benefit).  
**All 32 decoder layers + lm_head** are INT4.  
**Expected output**: ~2–2.5 GB total (vs ~9 GB FP16).

In [ ]:
result = subprocess.run(
    [
        sys.executable, str(SCRIPTS_DIR / "convert_to_openvino.py"),
        "--model-dir", str(MODEL_DIR),
        "--output",    str(OUTPUT_DIR),
        "--dtype",     "bf16",
        "--group-size", "64",   # INT4 group size (64 = better quality, 128 = faster)
        # "--no-int4",           # uncomment to get FP16 output instead
        # "--compile-check",     # uncomment to verify every IR loads on CPU
    ],
    cwd=str(QWEN35_DIR),
    env={**os.environ},
)
if result.returncode != 0:
    raise RuntimeError("convert_to_openvino.py failed.")
print("\n✓ INT4 Split-IR saved to", OUTPUT_DIR)

# Quick sanity check
xmls = sorted(OUTPUT_DIR.glob("*.xml"))
bins = sorted(OUTPUT_DIR.glob("*.bin"))
total_mb = sum(b.stat().st_size for b in bins) / 1e6
print(f"  {len(xmls)} IR files, {total_mb:.0f} MB total weights")

## 6 · Push to HuggingFace

In [ ]:
result = subprocess.run(
    [
        sys.executable, str(SCRIPTS_DIR / "push_to_hf.py"),
        "--ir-dir",    str(OUTPUT_DIR),
        "--repo-name", HF_REPO_NAME,
        "--token",     HF_TOKEN,
    ],
    cwd=str(QWEN35_DIR),
    env={**os.environ},
)
if result.returncode != 0:
    raise RuntimeError("push_to_hf.py failed.")
print("\n✓ Uploaded to HuggingFace:", HF_REPO_NAME)